In [17]:
from qiskit_nature.second_q.drivers import PySCFDriver
from rdkit import Chem
from rdkit.Chem import AllChem
import requests

def formula_to_pyscf_geometry(formula: str) -> str:
    # FIX 1: Use the correct, full PubChem API domain endpoint
    url = f"https://nih.gov/{formula}/SMILES/JSON"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise ValueError(f"Could not find a valid chemical structure for formula: {formula}")
    
    # Extract the first matching SMILES string
    data = response.json()
    
    # FIX 2: Correctly access the list elements inside the PropertyTable dictionary
    smiles = data['PropertyTable']['Properties'][0]['SMILES']
    print(f"Formula '{formula}' resolved to SMILES: {smiles}")
    
    # 2. Convert SMILES to RDKit Molecule object and add explicit hydrogens
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Failed to parse SMILES string into an RDKit molecule.")
    mol = Chem.AddHs(mol)
    
    # 3. Embed molecule into 3D space using experimental torsion preferences
    status = AllChem.EmbedMolecule(mol, useExpTorsionAnglePrefs=True, useBasicKnowledge=True)
    if status == -1:
        raise RuntimeError("3D conformation generation failed.")
        
    # Optimize geometry using Universal Force Field (UFF) for cleaner coordinates
    AllChem.UFFOptimizeMolecule(mol)
    
    # 4. Extract atomic symbols and 3D positions
    atom_lines = []
    conformer = mol.GetConformer()
    for i, atom in enumerate(mol.GetAtoms()):
        symbol = atom.GetSymbol()
        pos = conformer.GetAtomPosition(i)
        atom_lines.append(f"{symbol} {pos.x:.5f} {pos.y:.5f} {pos.z:.5f}")
    
    # Join into a semicolon-separated PySCFDriver format
    return "; ".join(atom_lines)

In [ ]:
# Input your desired chemical formula
target_formula = "H2O" 

try:
    pyscf_geometry = formula_to_pyscf_geometry(target_formula)
    print("\nGenerated Geometry String:")
    print(pyscf_geometry)
    
    # 5. Feed directly into Qiskit's PySCFDriver
    driver = PySCFDriver(atom=pyscf_geometry, basis="sto3g")
    problem = driver.run()
    print("\nPySCFDriver successfully initialized!")
    
except Exception as e:
    print(f"Error: {e}")

Error: Could not find a valid chemical structure for formula: H2O
